# Phase 0 + 1: Why Math is Hard for AI, and Chain-of-Thought Prompting

**Competition:** AI Mathematical Olympiad — Progress Prize 3  
**Goal of this notebook:** Understand *why* pure LLM generation fails at math, and show how Chain-of-Thought (CoT) prompting improves it.  

---

## What are we trying to do?

The competition asks an AI to solve problems like this:

> *Let $P(x) = x^3 + ax^2 + bx + c$ be a polynomial with integer coefficients. Given that $P(1) = 2$, $P(2) = 3$, $P(3) = 4$, find $P(4)$.*

The answer must be a 5-digit integer (or padded to 5 digits). With 110 such problems at IMO difficulty, this is one of the hardest open problems in AI today.

---

## Phase 0 — Why is Math Hard for LLMs?

### What an LLM actually does

An LLM (Large Language Model) is, at its core, a **next-token predictor**. Given the sequence of tokens so far, it outputs a probability distribution over what token comes next. It picks (or samples) from that distribution and repeats.

This is a powerful capability — it means the model has absorbed enormous amounts of human knowledge from text. But it has a fatal flaw for math:

> **The model generates what *looks plausible*, not what is *logically correct*.**

In natural language, plausible ≈ correct most of the time. In mathematics, plausible ≠ correct — even a single arithmetic error cascades into a wrong answer.

### The three failure modes

**1. Arithmetic hallucination**  
The model "knows" that large multiplications produce large numbers, but it has no calculator. It generates a number that *looks right in context* but is wrong. Example:
```
37 × 84 = 3108   ← model output (wrong; correct is 3108... wait, actually 3108 is right!)
37 × 84 = 3208   ← but this is what models often say
```
Arithmetic is brittle because the training data has far more examples of small multiplications than large ones.

**2. No working memory**  
Math proofs require holding intermediate results in mind across many steps. The model's "memory" is just the context window — it sees tokens, not variables. It can lose track of which variable equals what.

**3. No ability to verify**  
A human can check: *"Does my answer satisfy the original equation?"* A pure text-generating LLM generates a final answer token and stops. It cannot plug the answer back in and check — unless you explicitly teach it to.

### Why code execution is the key unlock

If the model writes **Python code** and we **execute** it:
- Python's arithmetic is exact — no hallucination
- Variables are stored in memory — no tracking errors  
- We can verify by running the check as code
- `sympy` can do symbolic algebra, calculus, number theory

This is the fundamental insight behind Tool-Integrated Reasoning (TIR), which we'll build in notebook 2. For now, we start simpler.

---

## Phase 1 — Chain-of-Thought (CoT) Prompting

### What is CoT?

Chain-of-Thought prompting is the idea that if you instruct the model to *show its reasoning steps* before giving the final answer, it performs much better on complex tasks.

The canonical zero-shot trigger: **"Let's think step by step."**

**Why does it work?**  
By generating intermediate steps, the model conditions its next-token prediction on those steps. Each reasoning step constrains the space of plausible next tokens, steering away from shortcuts and toward logical deduction. In other words: writing the steps out is how the model "thinks".

**Without CoT:**
```
Q: If 5 machines make 5 widgets in 5 minutes, how many minutes does it take 
   100 machines to make 100 widgets?
A: 100 minutes   ← wrong (common intuitive trap)
```

**With CoT:**
```
Q: ... Let's think step by step.
A: Each machine makes 1 widget in 5 minutes.
   100 machines each make 1 widget in 5 minutes.
   So 100 machines make 100 widgets in 5 minutes.
   Answer: 5   ← correct
```

### Types of CoT

| Type | Description | When to use |
|---|---|---|
| **Zero-shot CoT** | Just add "Think step by step" to the prompt | Quick baseline |
| **Few-shot CoT** | Provide 2-4 worked examples in the prompt | Better accuracy |
| **Self-consistency** | Sample CoT N times, majority vote | Best accuracy |

In this notebook we explore zero-shot and few-shot CoT. Self-consistency comes in notebook 3.

---

## Setup

### Model choice: `Qwen/Qwen2.5-Math-7B-Instruct`

**Why this model?**
- **Math-specialized:** Qwen2.5-Math was pretrained on large math corpora (textbooks, papers, competition problems), then instruction-tuned. It dramatically outperforms general-purpose 7B models on math benchmarks.
- **7B parameters:** Fits on a single T4 GPU (free Kaggle/Colab tier) with 4-bit quantization, or comfortably on an A10G/A100.
- **Open weights:** Available on HuggingFace under an open license.
- **Strong baseline:** Scores ~75% on the MATH benchmark vs ~50% for general models of the same size.

### GPU requirement
- **Minimum:** T4 (16GB) with 4-bit quantization (Kaggle free tier)
- **Comfortable:** A10G or A100 (Kaggle P100 or Colab Pro)
- **CPU-only:** Very slow but works for 1-2 problems (not recommended)

### Install dependencies

In [ ]:
# Install required packages
# transformers: load and run HuggingFace models
# bitsandbytes: 4-bit quantization so the 7B model fits on T4 GPU
# accelerate: required by transformers for GPU dispatch
!pip install -q transformers bitsandbytes accelerate

### Load the model

We use **4-bit quantization** (`load_in_4bit=True`). This is a technique that reduces the precision of the model weights from 32-bit floats to 4-bit integers, cutting memory usage by ~8x with only a small accuracy loss. It's how we fit a 7B model into ~6GB of VRAM.

Key concept: **quantization** trades a tiny bit of quality for a lot of efficiency. For inference (not training), this tradeoff is almost always worth it.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Math-7B-Instruct"

# 4-bit quantization config
# - load_in_4bit: compress weights to 4 bits
# - bnb_4bit_compute_dtype: use bfloat16 for the actual matrix multiplications
# - bnb_4bit_use_double_quant: quantize the quantization constants too (saves a bit more memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",       # automatically place layers on GPU/CPU
    trust_remote_code=True,
)
model.eval()  # disable dropout, etc.

print("Model loaded!")
print(f"Device map: {model.hf_device_map}")

### Define a simple inference helper

Key parameters to understand:
- **`max_new_tokens`:** How many tokens the model is allowed to generate. Math solutions can be long — 1024 is a good starting point.
- **`temperature`:** Controls randomness. `temperature=0` → deterministic (always picks the most likely token). `temperature=0.7` → some randomness, useful for sampling diverse solutions later.
- **`do_sample`:** Must be `True` to use temperature > 0. If `False`, uses greedy decoding.

In [ ]:
def generate_response(prompt: str, max_new_tokens: int = 1024, temperature: float = 0.0) -> str:
    """
    Send a prompt to the model and return the generated text.
    
    Uses the model's chat template (Qwen2.5 is instruction-tuned,
    so it expects messages in a specific format).
    """
    messages = [{"role": "user", "content": prompt}]
    
    # apply_chat_template formats the messages into the token sequence
    # the model was trained to expect (e.g., <|im_start|>user\n...)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # adds the assistant turn opener
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():  # don't track gradients during inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the newly generated tokens (not the input prompt)
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

---

## Sample Problems

We use a small hand-picked problem set with **known integer answers**. These span the four competition topics and range from AMC-level to AIME-level difficulty. We'll measure how many the model gets right under each prompting strategy.

For the competition, the answers are 5 digits. Our problems here have smaller answers for readability — the principle is identical.

In [ ]:
# Each problem has:
#   'topic':  algebra / combinatorics / geometry / number_theory
#   'level':  AMC / AIME / IMO
#   'problem': the problem statement
#   'answer':  integer answer

problems = [
    {
        "id": 1,
        "topic": "algebra",
        "level": "AMC",
        "problem": (
            "If $x + y = 10$ and $xy = 20$, what is $x^2 + y^2$?"
        ),
        "answer": 60,
    },
    {
        "id": 2,
        "topic": "combinatorics",
        "level": "AMC",
        "problem": (
            "How many ways can 5 people be seated in a row "
            "if two specific people must NOT sit next to each other?"
        ),
        "answer": 72,
    },
    {
        "id": 3,
        "topic": "number_theory",
        "level": "AMC",
        "problem": (
            "Find the sum of all positive integers $n$ less than 100 "
            "such that $n$ is divisible by 3 but not by 9."
        ),
        "answer": 1116,
    },
    {
        "id": 4,
        "topic": "geometry",
        "level": "AMC",
        "problem": (
            "In a right triangle, the two legs have lengths 5 and 12. "
            "What is the area of the square drawn on the hypotenuse?"
        ),
        "answer": 169,
    },
    {
        "id": 5,
        "topic": "algebra",
        "level": "AIME",
        "problem": (
            "Let $a$, $b$, $c$ be positive real numbers satisfying $a + b + c = 1$. "
            "Find the minimum integer value of "
            "$\\frac{1}{a+b} + \\frac{1}{b+c} + \\frac{1}{a+c}$."
        ),
        # By AM-HM inequality the sum >= 9/2 = 4.5; infimum never reached,
        # so the minimum integer value the expression can exceed is 5.
        "answer": 5,
    },
    {
        "id": 6,
        "topic": "number_theory",
        "level": "AIME",
        "problem": (
            "Find the number of integers from 1 to 1000 (inclusive) "
            "that are divisible by at least one of 3, 5, or 7."
        ),
        # Inclusion-exclusion: 333+200+142 - 66-47-28 + 9 = 543
        "answer": 543,
    },
    {
        "id": 7,
        "topic": "combinatorics",
        "level": "AIME",
        "problem": (
            "A committee of 5 people is to be chosen from a group of 6 men and 4 women. "
            "How many committees include at least 2 women?"
        ),
        # Total C(10,5)=252. Subtract 0 women C(6,5)=6 and 1 woman C(4,1)*C(6,4)=60.
        # 252 - 6 - 60 = 186
        "answer": 186,
    },
]

print(f"Loaded {len(problems)} problems.")
for p in problems:
    print(f"  [{p['id']}] {p['level']:4s} | {p['topic']:15s} | Answer: {p['answer']}")

---

## Experiment 1: Baseline (No CoT)

We ask the model to answer directly, with no instruction to show reasoning. This is the simplest possible prompt.

**Expected outcome:** The model will get some AMC problems right by pattern matching, but will fail on harder ones. It has no mechanism to catch its own arithmetic errors.

In [ ]:
def make_baseline_prompt(problem_text: str) -> str:
    return (
        f"Solve this math problem. Your answer must be an integer.\n\n"
        f"Problem: {problem_text}\n\n"
        f"Answer: "
    )

def extract_integer_answer(text: str) -> int | None:
    """
    Extract the last integer found in the model output.
    Models often embed the answer in a sentence; we grab the last number.
    """
    import re
    # Find all integers (possibly negative) in the text
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    if not numbers:
        return None
    return int(numbers[-1])


print("=" * 60)
print("EXPERIMENT 1: BASELINE (NO COT)")
print("=" * 60)

baseline_results = []

for p in problems:
    prompt = make_baseline_prompt(p["problem"])
    response = generate_response(prompt, max_new_tokens=256, temperature=0.0)
    predicted = extract_integer_answer(response)
    correct = (predicted == p["answer"])
    
    baseline_results.append({
        "id": p["id"],
        "level": p["level"],
        "topic": p["topic"],
        "expected": p["answer"],
        "predicted": predicted,
        "correct": correct,
        "response": response,
    })
    
    status = "CORRECT" if correct else "WRONG"
    print(f"[{p['id']}] {p['level']:4s} | Expected: {p['answer']:6d} | Got: {str(predicted):6s} | {status}")
    print(f"       Response snippet: {response[:120].strip()}")
    print()

n_correct = sum(r["correct"] for r in baseline_results)
print(f"\nBaseline score: {n_correct}/{len(problems)}")

---

## Experiment 2: Zero-Shot CoT

We add **"Think step by step before giving the final answer."** to the prompt. That's the only change. We also change the output format to make it easier to extract the final answer.

**Expected outcome:** Noticeably better, especially on the combinatorics and number theory problems where multiple steps are required.

In [ ]:
def make_zeroshot_cot_prompt(problem_text: str) -> str:
    return (
        f"Solve this math problem. Think step by step before giving the final answer.\n"
        f"At the end, write your final integer answer on a new line as:\n"
        f"ANSWER: <integer>\n\n"
        f"Problem: {problem_text}\n\n"
        f"Solution:"
    )

def extract_labeled_answer(text: str) -> int | None:
    """
    Look for 'ANSWER: <number>' in the model's output.
    Fall back to extracting the last integer if the label is missing.
    """
    import re
    match = re.search(r"ANSWER:\s*(-?\d+)", text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    # Fallback
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    return int(numbers[-1]) if numbers else None


print("=" * 60)
print("EXPERIMENT 2: ZERO-SHOT COT")
print("=" * 60)

zs_cot_results = []

for p in problems:
    prompt = make_zeroshot_cot_prompt(p["problem"])
    response = generate_response(prompt, max_new_tokens=1024, temperature=0.0)
    predicted = extract_labeled_answer(response)
    correct = (predicted == p["answer"])
    
    zs_cot_results.append({
        "id": p["id"],
        "level": p["level"],
        "topic": p["topic"],
        "expected": p["answer"],
        "predicted": predicted,
        "correct": correct,
        "response": response,
    })
    
    status = "CORRECT" if correct else "WRONG"
    print(f"[{p['id']}] {p['level']:4s} | Expected: {p['answer']:6d} | Got: {str(predicted):6s} | {status}")
    print()

n_correct = sum(r["correct"] for r in zs_cot_results)
print(f"\nZero-shot CoT score: {n_correct}/{len(problems)}")

---

## Experiment 3: Few-Shot CoT

We provide **2 worked examples** in the prompt before the actual question. This is called "in-context learning" — the model observes the pattern (problem → step-by-step solution → ANSWER) and applies it.

**Why does this help beyond zero-shot CoT?**  
The examples anchor the *style* of reasoning. For math, they show: how to set up equations, how to name variables, how to structure the work. The model's generation is then constrained to follow that style.

**Important:** We use examples from **different topics** than the test problem, so we're not just showing the answer — we're showing the *process*.

In [ ]:
FEW_SHOT_EXAMPLES = """\
Problem: A train travels 120 km in 2 hours, then 180 km in 3 hours. What is its average speed over the entire journey?
Solution:
Step 1: Total distance = 120 + 180 = 300 km
Step 2: Total time = 2 + 3 = 5 hours
Step 3: Average speed = total distance / total time = 300 / 5 = 60 km/h
ANSWER: 60

Problem: How many 3-digit numbers have all distinct digits?
Solution:
Step 1: The hundreds digit can be 1–9 (not 0): 9 choices.
Step 2: The tens digit can be 0–9 except the hundreds digit: 9 choices.
Step 3: The units digit can be 0–9 except the two already chosen: 8 choices.
Step 4: Total = 9 × 9 × 8 = 648.
ANSWER: 648
"""

def make_fewshot_cot_prompt(problem_text: str) -> str:
    return (
        f"Solve math problems step by step. Write ANSWER: <integer> at the end.\n\n"
        f"{FEW_SHOT_EXAMPLES}\n"
        f"Problem: {problem_text}\n"
        f"Solution:"
    )


print("=" * 60)
print("EXPERIMENT 3: FEW-SHOT COT")
print("=" * 60)

fs_cot_results = []

for p in problems:
    prompt = make_fewshot_cot_prompt(p["problem"])
    response = generate_response(prompt, max_new_tokens=1024, temperature=0.0)
    predicted = extract_labeled_answer(response)
    correct = (predicted == p["answer"])
    
    fs_cot_results.append({
        "id": p["id"],
        "level": p["level"],
        "topic": p["topic"],
        "expected": p["answer"],
        "predicted": predicted,
        "correct": correct,
        "response": response,
    })
    
    status = "CORRECT" if correct else "WRONG"
    print(f"[{p['id']}] {p['level']:4s} | Expected: {p['answer']:6d} | Got: {str(predicted):6s} | {status}")
    print()

n_correct = sum(r["correct"] for r in fs_cot_results)
print(f"\nFew-shot CoT score: {n_correct}/{len(problems)}")

---

## Results Comparison

In [ ]:
print("=" * 70)
print(f"{'Problem':<10} {'Level':<6} {'Topic':<16} {'Expected':<10} {'Baseline':<10} {'ZS-CoT':<10} {'FS-CoT':<10}")
print("-" * 70)

for i, p in enumerate(problems):
    b = "✓" if baseline_results[i]["correct"] else "✗"
    z = "✓" if zs_cot_results[i]["correct"] else "✗"
    f = "✓" if fs_cot_results[i]["correct"] else "✗"
    print(f"{p['id']:<10} {p['level']:<6} {p['topic']:<16} {p['answer']:<10} {b:<10} {z:<10} {f:<10}")

print("-" * 70)
bl = sum(r["correct"] for r in baseline_results)
zs = sum(r["correct"] for r in zs_cot_results)
fs = sum(r["correct"] for r in fs_cot_results)
n  = len(problems)
print(f"{'TOTAL':<33} {bl}/{n}      {zs}/{n}      {fs}/{n}")
print("=" * 70)

---

## Inspect the Failures

This is the most important part. For every problem where the model failed, read the full response and understand *why* it went wrong. This directly informs what we build next.

In [ ]:
print("FAILURE ANALYSIS — Few-Shot CoT Errors")
print("=" * 60)

for i, p in enumerate(problems):
    r = fs_cot_results[i]
    if not r["correct"]:
        print(f"\nProblem {p['id']} ({p['level']}, {p['topic']}):")
        print(f"  Problem : {p['problem']}")
        print(f"  Expected: {p['answer']}")
        print(f"  Got     : {r['predicted']}")
        print(f"  Model output:")
        print("  " + r["response"].replace("\n", "\n  "))
        print()

failures = [r for r in fs_cot_results if not r["correct"]]
if not failures:
    print("No failures! Try harder problems or move to notebook 2.")

---

## What Did We Learn?

### Key observations (fill in after running)

1. **Baseline vs CoT gap:** CoT improved accuracy by ___ problems. The reasoning traces make the difference visible — the model's error on problem X was ___.

2. **Where CoT still fails:** Hard AIME/IMO problems. Specifically:
   - Arithmetic: even with step-by-step, the model can compute `a × b` wrong
   - Multi-case reasoning: problems requiring exhaustive enumeration tend to miss cases
   - Algebraic manipulation: symbolic errors accumulate over many steps

3. **The ceiling of pure text reasoning:** CoT is powerful but has a hard ceiling. The model generating reasoning in text is fundamentally different from executing code. It cannot verify its own arithmetic.

### What comes next (Notebook 2)

The fix for all of the above: **let the model write Python code and execute it**.

- Arithmetic errors → Python computes exactly
- Enumeration → `for` loops enumerate exhaustively
- Symbolic algebra → `sympy` does it symbolically
- Self-verification → the model can write an assertion check in code

This is **Tool-Integrated Reasoning (TIR)**, the core technique of the AIMO winners.

---

## Exercises (before moving to notebook 2)

1. **Add a harder problem** — find an AIME problem online (with integer answer), add it to `problems`, and see how the model does.

2. **Try temperature > 0** — re-run one problem 5 times with `temperature=0.7`. How much do the answers vary? This variance is why majority voting (Phase 3) is important.

3. **Inspect the tokenization** — run `tokenizer.encode("37 × 84")` and look at how numbers are tokenized. Notice that multi-digit numbers are often split across tokens — this is a root cause of arithmetic errors.

4. **Try `sympy` manually** — open a Python shell and solve problem 1 with sympy:
   ```python
   from sympy import symbols, solve
   x, y = symbols('x y')
   # x + y = 10, xy = 20, find x^2 + y^2
   ```
   This is what the model will learn to generate in notebook 2.